In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
from langchain_core.tools import tool


In [2]:
from langchain_tavily import TavilySearch
tavily_search = TavilySearch(
    max_results=2,
    topic="general",
    include_answer=True
)
@tool
def web_search(query:str) -> str:

    """
    Uses tavily tool to search the web for up-to-date information and returns relevant results with sources.
    """

    return tavily_search.invoke({"query":query})['answer']
# web_search("what is recent ai news")

In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)


c:\Users\soumy\Desktop\GenAIEngineer\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

# Files are saved in langgraph state

# By default we provide a StateBackend
agent = create_deep_agent(model=llm)

# Under the hood, it looks like
agent = create_deep_agent(
    model=llm,
    backend=StateBackend(),
)

In [5]:
result=agent.invoke({
    "messages":[{
        "role":"user","content":(
            "Create a file at /notes/todo.txt with exactly the below content:\n"
            "1. Record video\n"
            "2. Edit video\n"
            "3.Upload video\n"
            "Then tell me you have done it."
        )
    }]
})

In [6]:
result

{'messages': [HumanMessage(content='Create a file at /notes/todo.txt with exactly the below content:\n1. Record video\n2. Edit video\n3.Upload video\nThen tell me you have done it.', additional_kwargs={}, response_metadata={}, id='5b6b4b31-131f-4cd2-9348-1e74a981c190'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '71qhhm66q', 'function': {'arguments': '{"content":"1. Record video\\n2. Edit video\\n3. Upload video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 3256, 'total_tokens': 3293, 'completion_time': 0.104447543, 'completion_tokens_details': None, 'prompt_time': 0.200891907, 'prompt_tokens_details': None, 'queue_time': 0.059005112, 'total_time': 0.30533945}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--

In [7]:
files=result.get('files',{})

if files:
    print(f"{len(files)}")
    for path,content in files.items():
        print(f"{path}\n{'-'*40}\n{content}")
else :
    print("No files found in state. Either the agent did not write a file, or the backend isn't wired up correctly")

1
/notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-08-08T08:56:42.136526+00:00', 'modified_at': '2026-08-08T08:56:42.136526+00:00'}


In [14]:
followup=agent.invoke({
    "messages":result["messages"]+[{
        "role":"user",
        "content":"Read /notes/todo.txt back to me."
    }],
    "files":result.get("files",{})
    # pass the cirtual filesystem along
})

followup

{'messages': [HumanMessage(content='Create a file at /notes/todo.txt with exactly the below content:\n1. Record video\n2. Edit video\n3.Upload video\nThen tell me you have done it.', additional_kwargs={}, response_metadata={}, id='5b6b4b31-131f-4cd2-9348-1e74a981c190'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '71qhhm66q', 'function': {'arguments': '{"content":"1. Record video\\n2. Edit video\\n3. Upload video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 3256, 'total_tokens': 3293, 'completion_time': 0.104447543, 'completion_tokens_details': None, 'prompt_time': 0.200891907, 'prompt_tokens_details': None, 'queue_time': 0.059005112, 'total_time': 0.30533945}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--

In [15]:
print(followup["messages"][-1].content)

The contents of /notes/todo.txt are:

1. Record video
2. Edit video
3. Upload video


In [29]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
ROOT="."
agent = create_deep_agent(
    model=llm,
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

In [30]:
result=agent.invoke({
    "messages":[{
        "role":"user","content":(
            "Create a file at /notes/todo.txt with exactly the below content:\n"
            "1. Record video\n"
            "2. Edit video\n"
            "3.Upload video\n"
            "Then tell me you have done it."
        )
    }]
})

In [18]:
result

{'messages': [HumanMessage(content='Create a file at /notes/todo.txt with exactly the below content:\n1. Record video\n2. Edit video\n3.Upload video\nThen tell me you have done it.', additional_kwargs={}, response_metadata={}, id='c6d98bc0-feb5-4014-95fc-fd098ae367fc'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ytmmwttzt', 'function': {'arguments': '{"content":"1. Record video\\n2. Edit video\\n3. Upload video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 3256, 'total_tokens': 3293, 'completion_time': 0.092596972, 'completion_tokens_details': None, 'prompt_time': 0.187039287, 'prompt_tokens_details': None, 'queue_time': 0.058645509, 'total_time': 0.279636259}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run-

In [31]:
from pathlib import Path
# disk_path=Path(ROOT)
disk_path=Path("./notes/todo.txt")

if disk_path.exists():
    print("File exists on disk")
    print(disk_path.resolve())
    print(disk_path.read_text())
else:
    print(f"expected file not at {disk_path.resolve()}")
    print("The agent may ot have called the write tool, or the path mapping differs")


File exists on disk
C:\Users\soumy\Desktop\GenAIEngineer\DeepAgents\notes\todo.txt
1. Record video
2. Edit video
3. Upload video


In [ ]:
followup=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Read /notes/todo.txt back to me verbatim."
    }],
    # pass the cirtual filesystem along
})

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=read_file {"file_path": "/notes/todo.txt", "limit": 100, "offset": 0}</function>'}}